# 0. Import library

In [13]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("cuda is available")
else:
    print("cuda is NOT available")

import numpy as np
from tqdm import tqdm
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import time
import copy
from moving_average import moving_average_1d

import importlib
import policy
importlib.reload(policy)
from policy import PolicyNN

from nn_functions import surrogate

import sys
sys.path.append('../1_model')
from TiDE import TideModule, quantile_loss  


cuda is available


# 1. Organize data

In [14]:
class scalers():
    def __init__(self,x_max, x_min, y_max, y_min) -> None:
        self.x_max = x_max
        self.x_min = x_min
        self.y_max = y_max
        self.y_min = y_min
        
        return None
    
    def scaler_x(self, x_original, dim_id = -1):
        if dim_id == -1:
            x_s = -1 + 2 * ((x_original - self.x_min) / (self.x_max-self.x_min))
            return x_s
        else: 
            x_s = -1 + 2 * (x_original - self.x_min[0,dim_id]) / (self.x_max[0,dim_id] - self.x_min[0,dim_id])
            return x_s
    
    def inv_scaler_x(self, x_s, dim_id = -1):
        
        if dim_id == -1:
            x_original = (x_s + 1)*0.5*(self.x_max-self.x_min) + self.x_min
            return x_original
        else: 
            x_original = (x_s + 1)*0.5*(self.x_max[0,dim_id] - self.x_min[0,dim_id]) + self.x_min[0,dim_id]
            return x_original
        
    def scaler_y(self, y_original):
        return -1 + 2 * ((y_original - self.y_min) / (self.y_max-self.y_min))
    
    def inv_scaler_y(self, y_s):
        return (y_s + 1)*0.5*(self.y_max-self.y_min) + self.y_min

In [15]:
import numpy as np
import pandas as pd
import torch
import copy
from tqdm import tqdm

# =====================================================
# 1. Load & preprocess raw data
# =====================================================
df_all = pd.read_csv('../0_data/merged_df_2_99_temp_depth.csv')

print(df_all.shape)
print(df_all.columns)

# ---- global time index (핵심) ----
df_all = df_all.reset_index(drop=True)
df_all["time_id"] = np.arange(len(df_all))

# ---- NaN handling (time_id 유지) ----
nan_rows = df_all[df_all.isna().any(axis=1)]
df_all = df_all.dropna().reset_index(drop=True)

# =====================================================
# 2. Feature extraction
# =====================================================
loc_Z = df_all["Z"].to_numpy().reshape(-1, 1)
dist_X = df_all["Dist_to_nearest_X"].to_numpy().reshape(-1, 1)
dist_Y = df_all["Dist_to_nearest_Y"].to_numpy().reshape(-1, 1)
laser_power = df_all["Laser_power"].to_numpy().reshape(-1, 1)

# ---- global time id ----
time_id_all = df_all["time_id"].to_numpy()

# =====================================================
# 3. Melt pool smoothing
# =====================================================
mp_temp_raw = df_all["melt_pool_temperature"].to_numpy()
mp_temp_mv = moving_average_1d(mp_temp_raw, 4)
mp_temp = copy.deepcopy(mp_temp_raw)
mp_temp[1:-2] = mp_temp_mv
mp_temp = mp_temp.reshape(-1, 1)

mp_depth_raw = df_all["melt_pool_depth"].to_numpy()
mp_depth_mv = moving_average_1d(mp_depth_raw, 4)
mp_depth = copy.deepcopy(mp_depth_raw)
mp_depth[1:-2] = mp_depth_mv
mp_depth = mp_depth.reshape(-1, 1)

# =====================================================
# 4. Stack original-scale arrays
# =====================================================
x_original = np.concatenate((loc_Z, dist_X, dist_Y, laser_power), axis=1)  # (N,4)
y_original = np.concatenate((mp_temp, mp_depth), axis=1)                  # (N,2)

# =====================================================
# 5. Scaling (-1 ~ 1)
# =====================================================
x_max = np.max(x_original, axis=0, keepdims=True)
x_min = np.min(x_original, axis=0, keepdims=True)
y_max = np.max(y_original, axis=0, keepdims=True)
y_min = np.min(y_original, axis=0, keepdims=True)

scaler = scalers(x_max, x_min, y_max, y_min)

x_s = scaler.scaler_x(x_original)
y_s = scaler.scaler_y(y_original)

# =====================================================
# 6. Reference / Constraint (global arrays)
# =====================================================
length = y_s.shape[0]

# =====================================================
# Global reference (temperature)
# =====================================================
y_ref_global = y_s[:, 0:1]   # (N, 1)


# depth constraint (global)
e = 1e-4
y_depth_low = np.random.uniform(0.1423 - e, 0.1423 + e, size=(length, 1))
y_depth_up  = np.random.uniform(0.4126 - e, 0.4126 + e, size=(length, 1))
y_const_global = np.concatenate((y_depth_low, y_depth_up), axis=1)  # (N,2)

# =====================================================
# 7. Train / Validation split (time-consistent)
# =====================================================
cutoff_index = int(np.round(0.8 * length))

x_train, y_train = x_s[:cutoff_index], y_s[:cutoff_index]
x_val,   y_val   = x_s[cutoff_index:], y_s[cutoff_index:]

time_id_train = time_id_all[:cutoff_index]
time_id_val   = time_id_all[cutoff_index:]

# =====================================================
# 8. Sliding window construction (WITH time index)
# =====================================================
window = 50
P = 50

# ------------------ Training set ------------------
n_train = cutoff_index - window - P

x_past_train = np.empty((n_train, window, 4))
y_past_train = np.empty((n_train, window, 2))
x_future_train = np.empty((n_train, P, 3))
y_ref_train_seq = np.empty((n_train, P, 1))
y_const_train_seq = np.empty((n_train, P, 2))
time_idx_train = np.empty(n_train, dtype=np.int64)

for i in tqdm(range(window, cutoff_index - P)):
    j = i - window

    x_past_train[j] = x_train[i-window:i]
    y_past_train[j] = y_train[i-window:i]
    x_future_train[j] = x_train[i:i+P, :3]

    # reference & constraint (initial step1)
    y_ref_train_seq[j] = y_ref_global[i:i+P]
    y_const_train_seq[j] = y_const_global[i:i+P]

    # ⭐ global time index
    time_idx_train[j] = time_id_train[i]

# ------------------ Validation set ------------------
val_len = length - cutoff_index
n_val = val_len - window - P

x_past_val = np.empty((n_val, window, 4))
y_past_val = np.empty((n_val, window, 2))
x_future_val = np.empty((n_val, P, 3))
y_ref_val_seq = np.empty((n_val, P, 1))
y_const_val_seq = np.empty((n_val, P, 2))
time_idx_val = np.empty(n_val, dtype=np.int64)

for i in tqdm(range(window, val_len - P)):
    j = i - window

    x_past_val[j] = x_val[i-window:i]
    y_past_val[j] = y_val[i-window:i]
    x_future_val[j] = x_val[i:i+P, :3]

    y_ref_val_seq[j] = y_ref_global[cutoff_index + i : cutoff_index + i + P]
    y_const_val_seq[j] = y_const_global[cutoff_index + i : cutoff_index + i + P]

    time_idx_val[j] = time_id_val[i]

# =====================================================
# 9. Torch tensor conversion
# =====================================================
x_past_train = torch.tensor(x_past_train, dtype=torch.float32)
y_past_train = torch.tensor(y_past_train, dtype=torch.float32)
x_future_train = torch.tensor(x_future_train, dtype=torch.float32)
y_ref_train_seq = torch.tensor(y_ref_train_seq, dtype=torch.float32)
y_const_train_seq = torch.tensor(y_const_train_seq, dtype=torch.float32)
time_idx_train = torch.tensor(time_idx_train, dtype=torch.long)

x_past_val = torch.tensor(x_past_val, dtype=torch.float32)
y_past_val = torch.tensor(y_past_val, dtype=torch.float32)
x_future_val = torch.tensor(x_future_val, dtype=torch.float32)
y_ref_val_seq = torch.tensor(y_ref_val_seq, dtype=torch.float32)
y_const_val_seq = torch.tensor(y_const_val_seq, dtype=torch.float32)
time_idx_val = torch.tensor(time_idx_val, dtype=torch.long)

# =====================================================
# 10. Shape check
# =====================================================
print("Train shapes:")
print("x_past:", x_past_train.shape)
print("y_past:", y_past_train.shape)
print("x_future:", x_future_train.shape)
print("y_ref:", y_ref_train_seq.shape)
print("y_const:", y_const_train_seq.shape)
print("time_idx:", time_idx_train.shape)

print("\nVal shapes:")
print("x_past:", x_past_val.shape)
print("y_past:", y_past_val.shape)
print("x_future:", x_future_val.shape)
print("y_ref:", y_ref_val_seq.shape)
print("y_const:", y_const_val_seq.shape)
print("time_idx:", time_idx_val.shape)


(610615, 12)
Index(['time_index', 'melt_pool_temperature', 'melt_pool_depth',
       'scanning_speed', 'X', 'Y', 'Z', 'Dist_to_nearest_X',
       'Dist_to_nearest_Y', 'Dist_to_nearest_Z', 'Laser_power',
       'laser_power_number'],
      dtype='object')


100%|██████████| 121983/121983 [00:00<00:00, 214030.36it/s]


Train shapes:
x_past: torch.Size([488234, 50, 4])
y_past: torch.Size([488234, 50, 2])
x_future: torch.Size([488234, 50, 3])
y_ref: torch.Size([488234, 50, 1])
y_const: torch.Size([488234, 50, 2])
time_idx: torch.Size([488234])

Val shapes:
x_past: torch.Size([121983, 50, 4])
y_past: torch.Size([121983, 50, 2])
x_future: torch.Size([121983, 50, 3])
y_ref: torch.Size([121983, 50, 1])
y_const: torch.Size([121983, 50, 2])
time_idx: torch.Size([121983])


In [16]:
batch_size = 256

# ------------------ Training loader ------------------
train_dataset = TensorDataset(
    x_past_train,
    y_past_train,
    x_future_train,
    y_ref_train_seq,
    y_const_train_seq,
    time_idx_train          # ⭐ 반드시 포함
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False           # ⭐ 핵심 수정
)

# ------------------ Validation loader ------------------
val_dataset = TensorDataset(
    x_past_val,
    y_past_val,
    x_future_val,
    y_ref_val_seq,
    y_const_val_seq,
    time_idx_val
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


# 2. Loss function

In [17]:
def DPC_loss(
    x_past: torch.Tensor,
    tide_output: torch.Tensor,
    reference: torch.Tensor,
    u_output: torch.Tensor,
    constraint: torch.Tensor,
    w_tracking=1,
    w_smooth=1,
    w_constraint=1,
    debug=False
):

    tide_med = tide_output.median(dim=-1).values  # (B,T,2)
    tide_output_temp = tide_med[:, :, 0]
    tide_output_depth = tide_med[:, :, 1]

    reference = reference[:, :, 0]

    tracking_loss = (reference - tide_output_temp) ** 2

    #T = tracking_loss.shape[1]
    #q_weights = torch.ones((1, T), device=tracking_loss.device)
    #tracking_loss = tracking_loss * q_weights

    u_diff = u_output[:, 1:, :] - u_output[:, :-1, :]
    smoothness_loss = u_diff ** 2

    low_violation = torch.relu(constraint[:, :, 0] - tide_output_depth) ** 2
    up_violation  = torch.relu(tide_output_depth - constraint[:, :, 1]) ** 2
    constraint_loss = up_violation  # low_violation intentionally ignored

    # ====== mean values ======
    tracking_mean = tracking_loss.mean()
    smooth_mean = smoothness_loss.mean()
    constraint_mean = constraint_loss.mean()

    # ====== NaN debug ======
    if debug and (
        torch.isnan(tracking_mean)
        or torch.isnan(smooth_mean)
        or torch.isnan(constraint_mean)
    ):
        print("\n[NaN DEBUG in DPC_loss]")
        print("tracking_mean:", tracking_mean)
        print("smooth_mean:", smooth_mean)
        print("constraint_mean:", constraint_mean)

        print("tide_output_temp stats:",
              tide_output_temp.min().item(),
              tide_output_temp.max().item())

        print("tide_output_depth stats:",
              tide_output_depth.min().item(),
              tide_output_depth.max().item())

        print("constraint stats:",
              constraint.min().item(),
              constraint.max().item())

        print("u_output stats:",
              u_output.min().item(),
              u_output.max().item())

    L_tracking = torch.sqrt(tracking_mean)
    L_smooth = torch.sqrt(smooth_mean)
    L_constraint = torch.sqrt(constraint_mean)

    loss = (
        w_tracking * L_tracking +
        w_smooth * L_smooth +
        w_constraint * L_constraint
    )

    return loss, L_tracking.item(), L_smooth.item(), L_constraint.item()


# 3. Import TiDE model (NN model)

In [18]:
import torch
import pickle

# Load model
with open('TiDE_params_single_track_square_MV_temp_depth_less_cov_0915_w50_p50.pkl', 'rb') as file:
    nominal_params = pickle.load(file)

TiDE = nominal_params['model'].to(device)
total_params = sum(p.numel() for p in TiDE.parameters())

# 4. Train

In [19]:
class RolloutOneStepGenerator:
    @staticmethod
    def step(x_past, y_past, u0, yhat0):
        """
        x_past : (B, T, Dx)
        y_past : (B, T, Dy)
        u0     : (B, 1)
        yhat0  : (B, Dy)
        """

        # clone (절대 원본 수정 X)
        x_new = x_past.clone()
        y_new = y_past.clone()

        # shift
        x_new = torch.roll(x_new, shifts=-1, dims=1)
        y_new = torch.roll(y_new, shifts=-1, dims=1)

        # append
        x_new[:, -1, 3] = u0.squeeze(-1)
        y_new[:, -1, 0] = yhat0[:, 0]
        y_new[:, -1, 1] = yhat0[:, 1]

        return x_new, y_new


In [20]:
from torch.utils.data import Sampler

class ValidIndexSampler(Sampler):
    def __init__(self, valid_indices):
        self.valid_indices = valid_indices

    def __iter__(self):
        return iter(self.valid_indices)

    def __len__(self):
        return len(self.valid_indices)


In [21]:
import os
import copy
import random
import csv
import torch
from torch.utils.data import DataLoader, ConcatDataset, Dataset


# ==================================================
# Dataset wrapper
# ==================================================
class ListDataset(Dataset):
    def __init__(self, samples_list):
        self.samples = samples_list

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ==================================================
# Rollout (1-step)
# ==================================================
def _rollout_one_step_collect_candidates(model, loader):
    model.eval()
    candidates = []

    with torch.no_grad():
        for batch in loader:
            x_past, y_past, _, _, _, time_idx = batch

            x_past = x_past.to(device)
            y_past = y_past.to(device)
            time_idx = time_idx.to(device)

            next_idx = time_idx.cpu() + 1
            valid_mask = next_idx < x_future_train.shape[0]
            if not valid_mask.any():
                continue

            x_past = x_past[valid_mask]
            y_past = y_past[valid_mask]
            next_idx = next_idx[valid_mask]

            x_future_next = x_future_train[next_idx].cpu()
            y_ref_next = y_ref_train_seq[next_idx].cpu()
            y_const_next = y_const_train_seq[next_idx].cpu()

            u_output = model((
                torch.cat((x_past, y_past), dim=2),
                torch.cat((x_future_next.to(device),
                           y_ref_next.to(device),
                           y_const_next.to(device)), dim=2)
            ))

            tide_pred = TiDE((
                torch.cat((y_past, x_past), dim=2),
                torch.cat((x_future_next.to(device), u_output), dim=2),
                None
            ))

            u0 = u_output[:, 0, :]
            yhat0 = tide_pred.median(dim=-1).values[:, 0, :]

            x_past_next, y_past_next = RolloutOneStepGenerator.step(
                x_past, y_past, u0, yhat0
            )

            for i in range(x_past_next.size(0)):
                candidates.append((
                    x_past_next[i].cpu(),
                    y_past_next[i].cpu(),
                    x_future_next[i],
                    y_ref_next[i],
                    y_const_next[i],
                    next_idx[i].cpu()
                ))

    return candidates


# ==================================================
# Train n epochs (epoch마다 validation + scheduler.step)
#  - train loss list 반환
#  - best val loss / best model state 반환 (✅ best model을 val loss로 고르기 위해)
# ==================================================
def _train_policy_n_iter(
    model, optimizer, scheduler,
    train_loader, val_loader,
    w_tracking, w_smooth, w_constraint,
    num_epoch
):
    train_epoch_losses = []

    best_val_loss_epoch = float("inf")
    best_model_state_epoch = None

    for e in range(num_epoch):
        # ===== TRAIN =====
        model.train()
        epoch_loss = 0.0
        nb = 0

        for x_past, y_past, x_future, y_ref, y_const, _ in train_loader:
            x_past = x_past.to(device)
            y_past = y_past.to(device)
            x_future = x_future.to(device)
            y_ref = y_ref.to(device)
            y_const = y_const.to(device)

            u_output = model((
                torch.cat((x_past, y_past), dim=2),
                torch.cat((x_future, y_ref, y_const), dim=2)
            ))

            tide_pred = TiDE((
                torch.cat((y_past, x_past), dim=2),
                torch.cat((x_future, u_output), dim=2),
                None
            ))

            loss, _, _, _ = DPC_loss(
                x_past, tide_pred, y_ref, u_output, y_const,
                w_tracking=w_tracking,
                w_smooth=w_smooth,
                w_constraint=w_constraint
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            nb += 1

        epoch_loss /= max(1, nb)
        train_epoch_losses.append(epoch_loss)
        print(f"[TRAIN] epoch={e+1} loss={epoch_loss:.6f}")

        # ===== VALIDATION =====
        model.eval()
        val_loss = 0.0
        nb = 0

        with torch.no_grad():
            for x_past, y_past, x_future, y_ref, y_const, _ in val_loader:
                x_past = x_past.to(device)
                y_past = y_past.to(device)
                x_future = x_future.to(device)
                y_ref = y_ref.to(device)
                y_const = y_const.to(device)

                u_output = model((
                    torch.cat((x_past, y_past), dim=2),
                    torch.cat((x_future, y_ref, y_const), dim=2)
                ))

                tide_pred = TiDE((
                    torch.cat((y_past, x_past), dim=2),
                    torch.cat((x_future, u_output), dim=2),
                    None
                ))

                l, _, _, _ = DPC_loss(
                    x_past, tide_pred, y_ref, u_output, y_const,
                    w_tracking=w_tracking,
                    w_smooth=w_smooth,
                    w_constraint=w_constraint
                )

                val_loss += l.item()
                nb += 1

        val_loss /= max(1, nb)
        print(f"[VAL]   epoch={e+1} loss={val_loss:.6f}")

        # ✅ epoch 내부 best (val 기준) 저장
        if val_loss < best_val_loss_epoch:
            best_val_loss_epoch = val_loss
            best_model_state_epoch = copy.deepcopy(model.state_dict())

        # ✅ baseline (1)과 동일: epoch마다 scheduler.step()
        scheduler.step()

    return {
        "train_losses": train_epoch_losses,
        "best_val_loss": best_val_loss_epoch,
        "best_model_state": best_model_state_epoch,
    }


# ==================================================
# Main training loop
# ==================================================
def train_and_evaluate(
    n_layers,
    hidden_dim,
    w_tracking,
    w_smooth,
    w_constraint,
    num_epoch,
    rollout_iters,
    save_every,
    exp_dir,
    setting_name,
    m_per_rollout,
):
    os.makedirs(exp_dir, exist_ok=True)
    ckpt_dir = os.path.join(exp_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)

    model = PolicyNN(
        past_input_dim=6,
        future_input_dim=6,
        output_dim=1,
        p=50,
        window=50,
        hidden_dim=hidden_dim,
        n_layers=n_layers,
        dropout_p=0.1
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=10, gamma=0.90
    )

    base_dataset = train_dataset
    aug_samples = []
    aug_dataset = ListDataset(aug_samples)

    csv_path = os.path.join(exp_dir, f"{setting_name}_train_loss.csv")
    if not os.path.exists(csv_path):
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=["global_step", "rollout", "epoch_in_rollout", "train_loss"]
            )
            writer.writeheader()

    global_step = 0
    best_model = None
    best_val_loss = float("inf")
    best_rollout_k = None
    last_model = None

    # INIT TRAIN
    init_loader = DataLoader(
        base_dataset,
        batch_size=batch_size,
        sampler=ValidIndexSampler(list(range(len(base_dataset)))),
        shuffle=False
    )

    init_out = _train_policy_n_iter(
        model, optimizer, scheduler,
        init_loader, val_loader,
        w_tracking, w_smooth, w_constraint,
        num_epoch
    )

    init_losses = init_out["train_losses"]

    # ✅ global best 갱신 (val 기준)
    if init_out["best_val_loss"] < best_val_loss:
        best_val_loss = init_out["best_val_loss"]
        best_model = init_out["best_model_state"]
        best_rollout_k = -1

    for e, loss in enumerate(init_losses):
        global_step += 1
        with open(csv_path, "a", newline="") as f:
            csv.DictWriter(
                f,
                fieldnames=["global_step", "rollout", "epoch_in_rollout", "train_loss"]
            ).writerow({
                "global_step": global_step,
                "rollout": -1,
                "epoch_in_rollout": e + 1,
                "train_loss": loss
            })

    # ROLLOUT LOOP
    for rollout_k in range(rollout_iters):
        print(f"\n===== Rollout {rollout_k} =====")

        cur_dataset = ConcatDataset([base_dataset, aug_dataset])
        cur_loader = DataLoader(
            cur_dataset,
            batch_size=batch_size,
            sampler=ValidIndexSampler(list(range(len(cur_dataset)))),
            shuffle=False
        )

        candidates = _rollout_one_step_collect_candidates(model, cur_loader)

        if len(candidates) > 0:
            m = min(m_per_rollout, len(candidates))
            aug_samples.extend(random.sample(candidates, m))
            print(f"[AUG] Added {m}, total aug={len(aug_samples)}")

        out = _train_policy_n_iter(
            model, optimizer, scheduler,
            cur_loader, val_loader,
            w_tracking, w_smooth, w_constraint,
            num_epoch
        )

        losses = out["train_losses"]

        # ✅ global best 갱신 (val 기준)
        if out["best_val_loss"] < best_val_loss:
            best_val_loss = out["best_val_loss"]
            best_model = out["best_model_state"]
            best_rollout_k = rollout_k

        for e, loss in enumerate(losses):
            global_step += 1
            with open(csv_path, "a", newline="") as f:
                csv.DictWriter(
                    f,
                    fieldnames=["global_step", "rollout", "epoch_in_rollout", "train_loss"]
                ).writerow({
                    "global_step": global_step,
                    "rollout": rollout_k,
                    "epoch_in_rollout": e + 1,
                    "train_loss": loss
                })

        last_model = copy.deepcopy(model.state_dict())

        # =========================
        # CHECKPOINT SAVE (rollout마다)
        # =========================
        if save_every and rollout_k % save_every == 0:
            ckpt_path = os.path.join(
                ckpt_dir, f"{setting_name}_rollout{rollout_k:04d}.pth"
            )
            torch.save(
                {
                    "rollout_k": rollout_k,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_val_loss_so_far": best_val_loss,
                    "best_rollout_k_so_far": best_rollout_k,
                    "num_aug_samples": len(aug_samples),
                },
                ckpt_path
            )
            print(f"[CKPT] Saved checkpoint: {ckpt_path}")

    # FINAL SAVE
    if best_model is None:
        # 이 경우는 사실상 없어야 하지만, 안전장치
        best_model = copy.deepcopy(last_model)
        best_rollout_k = best_rollout_k if best_rollout_k is not None else rollout_iters - 1

    torch.save(best_model, os.path.join(exp_dir, f"{setting_name}_best.pth"))
    torch.save(last_model, os.path.join(exp_dir, f"{setting_name}_last.pth"))

    return {
        "best_val_loss": best_val_loss,
        "best_rollout_iter": best_rollout_k,
        "num_aug_samples": len(aug_samples),
    }


# 초반을 baseline 처럼

In [22]:
import os
import copy
import random
import csv
import torch
from torch.utils.data import DataLoader, ConcatDataset, Dataset


# ==================================================
# Dataset wrapper
# ==================================================
class ListDataset(Dataset):
    def __init__(self, samples_list):
        self.samples = samples_list

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ==================================================
# Rollout (1-step)
# ==================================================
def _rollout_one_step_collect_candidates(model, loader):
    model.eval()
    candidates = []

    with torch.no_grad():
        for batch in loader:
            x_past, y_past, _, _, _, time_idx = batch

            x_past = x_past.to(device)
            y_past = y_past.to(device)
            time_idx = time_idx.to(device)

            next_idx = time_idx.cpu() + 1
            valid_mask = next_idx < x_future_train.shape[0]
            if not valid_mask.any():
                continue

            x_past = x_past[valid_mask]
            y_past = y_past[valid_mask]
            next_idx = next_idx[valid_mask]

            x_future_next = x_future_train[next_idx].cpu()
            y_ref_next = y_ref_train_seq[next_idx].cpu()
            y_const_next = y_const_train_seq[next_idx].cpu()

            u_output = model((
                torch.cat((x_past, y_past), dim=2),
                torch.cat((x_future_next.to(device),
                           y_ref_next.to(device),
                           y_const_next.to(device)), dim=2)
            ))

            tide_pred = TiDE((
                torch.cat((y_past, x_past), dim=2),
                torch.cat((x_future_next.to(device), u_output), dim=2),
                None
            ))

            u0 = u_output[:, 0, :]
            yhat0 = tide_pred.median(dim=-1).values[:, 0, :]

            x_past_next, y_past_next = RolloutOneStepGenerator.step(
                x_past, y_past, u0, yhat0
            )

            for i in range(x_past_next.size(0)):
                candidates.append((
                    x_past_next[i].cpu(),
                    y_past_next[i].cpu(),
                    x_future_next[i],
                    y_ref_next[i],
                    y_const_next[i],
                    next_idx[i].cpu()
                ))

    return candidates


# ==================================================
# Train n epochs
#  - train loss list 반환
#  - best val loss / best model state 반환
#  - (옵션) validation/scheduler/print 제어 가능
# ==================================================
def _train_policy_n_iter(
    model, optimizer, scheduler,
    train_loader, val_loader,
    w_tracking, w_smooth, w_constraint,
    num_epoch,
    do_validation=True,
    step_scheduler=True,
    verbose=False
):
    train_epoch_losses = []

    best_val_loss_epoch = float("inf")
    best_model_state_epoch = None

    for e in range(num_epoch):
        # ===== TRAIN =====
        model.train()
        epoch_loss = 0.0
        nb = 0

        for x_past, y_past, x_future, y_ref, y_const, _ in train_loader:
            x_past = x_past.to(device)
            y_past = y_past.to(device)
            x_future = x_future.to(device)
            y_ref = y_ref.to(device)
            y_const = y_const.to(device)

            u_output = model((
                torch.cat((x_past, y_past), dim=2),
                torch.cat((x_future, y_ref, y_const), dim=2)
            ))

            tide_pred = TiDE((
                torch.cat((y_past, x_past), dim=2),
                torch.cat((x_future, u_output), dim=2),
                None
            ))

            loss, _, _, _ = DPC_loss(
                x_past, tide_pred, y_ref, u_output, y_const,
                w_tracking=w_tracking,
                w_smooth=w_smooth,
                w_constraint=w_constraint
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            nb += 1

        epoch_loss /= max(1, nb)
        train_epoch_losses.append(epoch_loss)

        if verbose:
            print(f"[TRAIN] epoch={e+1} loss={epoch_loss:.6f}")

        # ===== VALIDATION (optional) =====
        if do_validation:
            model.eval()
            val_loss = 0.0
            nb = 0

            with torch.no_grad():
                for x_past, y_past, x_future, y_ref, y_const, _ in val_loader:
                    x_past = x_past.to(device)
                    y_past = y_past.to(device)
                    x_future = x_future.to(device)
                    y_ref = y_ref.to(device)
                    y_const = y_const.to(device)

                    u_output = model((
                        torch.cat((x_past, y_past), dim=2),
                        torch.cat((x_future, y_ref, y_const), dim=2)
                    ))

                    tide_pred = TiDE((
                        torch.cat((y_past, x_past), dim=2),
                        torch.cat((x_future, u_output), dim=2),
                        None
                    ))

                    l, _, _, _ = DPC_loss(
                        x_past, tide_pred, y_ref, u_output, y_const,
                        w_tracking=w_tracking,
                        w_smooth=w_smooth,
                        w_constraint=w_constraint
                    )

                    val_loss += l.item()
                    nb += 1

            val_loss /= max(1, nb)

            if verbose:
                print(f"[VAL]   epoch={e+1} loss={val_loss:.6f}")

            # ✅ epoch 내부 best (val 기준) 저장
            if val_loss < best_val_loss_epoch:
                best_val_loss_epoch = val_loss
                best_model_state_epoch = copy.deepcopy(model.state_dict())

        # ===== scheduler step (optional) =====
        if step_scheduler and scheduler is not None:
            scheduler.step()

    # validation을 껐으면, best_model_state_epoch가 None일 수 있음
    # -> train_and_evaluate에서 global best 업데이트를 INIT에서는 하지 않도록 처리할 거라 OK
    return {
        "train_losses": train_epoch_losses,
        "best_val_loss": best_val_loss_epoch,
        "best_model_state": best_model_state_epoch,
    }


# ==================================================
# Main training loop
# ==================================================
def train_and_evaluate(
    n_layers,
    hidden_dim,
    w_tracking,
    w_smooth,
    w_constraint,
    num_epoch,
    rollout_iters,
    save_every,
    exp_dir,
    setting_name,
    m_per_rollout,
):
    os.makedirs(exp_dir, exist_ok=True)
    ckpt_dir = os.path.join(exp_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)

    model = PolicyNN(
        past_input_dim=6,
        future_input_dim=6,
        output_dim=1,
        p=50,
        window=50,
        hidden_dim=hidden_dim,
        n_layers=n_layers,
        dropout_p=0.1
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=10, gamma=0.90
    )

    base_dataset = train_dataset
    aug_samples = []
    aug_dataset = ListDataset(aug_samples)

    csv_path = os.path.join(exp_dir, f"{setting_name}_train_loss.csv")
    if not os.path.exists(csv_path):
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=["global_step", "rollout", "epoch_in_rollout", "train_loss"]
            )
            writer.writeheader()

    global_step = 0
    best_model = None
    best_val_loss = float("inf")
    best_rollout_k = None
    last_model = None

    # ==================================================
    # INIT TRAIN (✅ baseline과 맞추기)
    #  - shuffle=True
    #  - sampler 제거
    #  - validation OFF
    #  - scheduler.step OFF
    # ==================================================
    init_loader = DataLoader(
        base_dataset,
        batch_size=batch_size,
        shuffle=False,   # ⭐ baseline처럼
        drop_last=False
    )

    init_out = _train_policy_n_iter(
        model, optimizer, scheduler,
        init_loader, val_loader,
        w_tracking, w_smooth, w_constraint,
        num_epoch,
        do_validation=False,     # ⭐ INIT에서는 val 끔
        step_scheduler=False,    # ⭐ INIT에서는 scheduler 끔
        verbose= True          # ⭐ 콘솔 loss 출력 끔
    )

    init_losses = init_out["train_losses"]

    for e, loss in enumerate(init_losses):
        global_step += 1
        with open(csv_path, "a", newline="") as f:
            csv.DictWriter(
                f,
                fieldnames=["global_step", "rollout", "epoch_in_rollout", "train_loss"]
            ).writerow({
                "global_step": global_step,
                "rollout": -1,
                "epoch_in_rollout": e + 1,
                "train_loss": loss
            })

    # ==================================================
    # ROLLOUT LOOP
    #  - validation ON
    #  - scheduler.step ON (epoch마다)
    #  - best model은 val 기준으로 갱신
    # ==================================================
    for rollout_k in range(rollout_iters):
        print(f"\n===== Rollout {rollout_k} =====")

        cur_dataset = ConcatDataset([base_dataset, aug_dataset])
        cur_loader = DataLoader(
            cur_dataset,
            batch_size=batch_size,
            sampler=ValidIndexSampler(list(range(len(cur_dataset)))),
            shuffle=False
        )

        candidates = _rollout_one_step_collect_candidates(model, cur_loader)

        if len(candidates) > 0:
            m = min(m_per_rollout, len(candidates))
            aug_samples.extend(random.sample(candidates, m))
            print(f"[AUG] Added {m}, total aug={len(aug_samples)}")

        out = _train_policy_n_iter(
            model, optimizer, scheduler,
            cur_loader, val_loader,
            w_tracking, w_smooth, w_constraint,
            num_epoch,
            do_validation=True,
            step_scheduler=True,
            verbose=True          # loss 출력 원하면 True
        )

        losses = out["train_losses"]

        # ✅ global best 갱신 (val 기준)
        if out["best_model_state"] is not None and out["best_val_loss"] < best_val_loss:
            best_val_loss = out["best_val_loss"]
            best_model = out["best_model_state"]
            best_rollout_k = rollout_k

        for e, loss in enumerate(losses):
            global_step += 1
            with open(csv_path, "a", newline="") as f:
                csv.DictWriter(
                    f,
                    fieldnames=["global_step", "rollout", "epoch_in_rollout", "train_loss"]
                ).writerow({
                    "global_step": global_step,
                    "rollout": rollout_k,
                    "epoch_in_rollout": e + 1,
                    "train_loss": loss
                })

        last_model = copy.deepcopy(model.state_dict())

        # =========================
        # CHECKPOINT SAVE (rollout마다)
        # =========================
        if save_every and rollout_k % save_every == 0:
            ckpt_path = os.path.join(
                ckpt_dir, f"{setting_name}_rollout{rollout_k:04d}.pth"
            )
            torch.save(
                {
                    "rollout_k": rollout_k,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "best_val_loss_so_far": best_val_loss,
                    "best_rollout_k_so_far": best_rollout_k,
                    "num_aug_samples": len(aug_samples),
                },
                ckpt_path
            )
            print(f"[CKPT] Saved checkpoint: {ckpt_path}")

    # FINAL SAVE
    if best_model is None:
        # INIT에서 val을 껐기 때문에, best_model은 rollout 과정에서만 정해짐.
        # rollout_iters=0 같은 경우 대비 안전장치
        best_model = copy.deepcopy(last_model)
        best_rollout_k = best_rollout_k if best_rollout_k is not None else -1
        best_val_loss = best_val_loss if best_val_loss != float("inf") else float("nan")

    torch.save(best_model, os.path.join(exp_dir, f"{setting_name}_best.pth"))
    torch.save(last_model, os.path.join(exp_dir, f"{setting_name}_last.pth"))

    return {
        "best_val_loss": best_val_loss,
        "best_rollout_iter": best_rollout_k,
        "num_aug_samples": len(aug_samples),
    }


In [23]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [24]:
batch_size = 256
learning_rate = 1e-3
weight_decay = 1e-5

results = train_and_evaluate(
    n_layers=3,
    hidden_dim=1024,
    w_tracking=1.0,
    w_smooth=1.0,
    w_constraint=1.0,
    num_epoch=5,          # n-iter
    rollout_iters=5,
    save_every=1,
    exp_dir="./experiments/rollout_aug_v1_rollout10_epoch10_t1_s1_c1_sampling1",
    setting_name="policy_aug_1step",
    m_per_rollout=1,
)


[TRAIN] epoch=1 loss=0.192729
[TRAIN] epoch=2 loss=0.110771
[TRAIN] epoch=3 loss=0.104749
[TRAIN] epoch=4 loss=0.099331
[TRAIN] epoch=5 loss=0.098411

===== Rollout 0 =====
[AUG] Added 1, total aug=1
[TRAIN] epoch=1 loss=0.096692
[VAL]   epoch=1 loss=0.105108
[TRAIN] epoch=2 loss=0.096649
[VAL]   epoch=2 loss=0.099117
[TRAIN] epoch=3 loss=0.095572
[VAL]   epoch=3 loss=0.097396
[TRAIN] epoch=4 loss=0.094405
[VAL]   epoch=4 loss=0.098356
[TRAIN] epoch=5 loss=0.092897
[VAL]   epoch=5 loss=0.097197
[CKPT] Saved checkpoint: ./experiments/rollout_aug_v1_rollout10_epoch10_t1_s1_c1_sampling1/checkpoints/policy_aug_1step_rollout0000.pth

===== Rollout 1 =====
[AUG] Added 1, total aug=2
[TRAIN] epoch=1 loss=0.092926
[VAL]   epoch=1 loss=0.096246
[TRAIN] epoch=2 loss=0.091993
[VAL]   epoch=2 loss=0.097576
[TRAIN] epoch=3 loss=0.091562
[VAL]   epoch=3 loss=0.096290
[TRAIN] epoch=4 loss=0.091031
[VAL]   epoch=4 loss=0.096773
[TRAIN] epoch=5 loss=0.090736
[VAL]   epoch=5 loss=0.093471
[CKPT] Saved c